# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaPrakash-Kaizu07/Flyrank-AI-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AdityaPrakash-Kaizu07/Flyrank-AI-intern"
REPO_DIR = "Flyrank-AI-intern"

if IN_COLAB:
    # In Colab, clone the repo if it doesn't exist yet
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

    # Install dependencies if you have a requirements.txt
    if os.path.exists("requirements.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # On a local machine, find the repo root from wherever this notebook started
    # (Checking for 'README.md' or another unique folder/file at your repo root)
    while not os.path.isfile("README.md") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())
# Ensure we are actually at the repo root by asserting a known file/folder exists
assert os.path.exists("README.md"), "Root file not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/Flyrank-AI-intern
Starter data found. You're ready.


In [4]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.0

In [5]:
# Step 1: Define your bucket edges (the ranges you want)
bucket_edges = [0, 30, 90, 180, float('inf')]
bucket_labels = ['0-30 days', '31-90 days', '91-180 days', '180+ days']

# Step 2: Use pd.cut() to assign each page to a bucket
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=bucket_edges,
    labels=bucket_labels,
    right=False  # means [0, 30) not (0, 30]
)

# Step 3: Group and count
staleness_table = df['staleness_bucket'].value_counts().sort_index()
print("STALENESS SIGNAL")
print(staleness_table)
print(f"Total pages: {len(df)}")

STALENESS SIGNAL
staleness_bucket
0-30 days      20480
31-90 days       175
91-180 days     9171
180+ days        174
Name: count, dtype: int64
Total pages: 30000


In [6]:
# Step 1: Use pd.qcut() with q=4 (4 quartiles: top 25%, 50-75%, 25-50%, bottom 25%)
df['traffic_bucket'] = pd.qcut(
    df['impressions_90d'],
    q=4,
    labels=['Bottom 25%', '25-50%', '50-75%', 'Top 25%'],
    duplicates='drop'  # handles ties gracefully
)

# Step 2: Group and show n + mean impressions per bucket
traffic_table = df.groupby('traffic_bucket', observed=True).agg({
    'impressions_90d': ['count', 'mean', 'min', 'max']
}).round(1)

print("TRAFFIC SIGNAL")
print(traffic_table)

TRAFFIC SIGNAL
               impressions_90d                       
                         count     mean   min     max
traffic_bucket                                       
Bottom 25%                7503     19.7     1      81
25-50%                    7499    334.8    82     731
50-75%                    7498   1785.3   732    3615
Top 25%                   7500  18662.2  3616  517715


In [8]:
# AND logic
and_candidates = df[
    (df['days_since_last_update'] >= 91) &
    (df['traffic_bucket'] == 'Bottom 25%')
].shape[0]

# OR logic
or_candidates = df[
    (df['days_since_last_update'] >= 91) |
    (df['traffic_bucket'] == 'Bottom 25%')
].shape[0]

print(f"AND candidates: {and_candidates}")
print(f"OR candidates: {or_candidates}")

AND candidates: 1138
OR candidates: 15710


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.